# LLM Enrichment Pipeline – Version 2 (IdiomX)

## Overview

This notebook implements **Version 2 of the IdiomX LLM enrichment pipeline**, which generates high-quality structured idiom examples using a large language model. The goal of this stage is to transform the raw idiom dataset into a **rich supervised training dataset** suitable for multiple NLP tasks including:

- Idiom detection
- Literal vs idiomatic usage classification
- Context → idiom prediction
- Arabic ↔ English idiom understanding
- Idiom semantic interpretation

The enrichment process expands each idiom entry with **structured metadata and multiple labeled examples**, enabling robust downstream modeling.

---

# Motivation for Version 2

The initial enrichment pipeline (V1) successfully generated examples but revealed several practical issues during dataset inspection and model training preparation:

- Occasional **invalid JSON responses** from the LLM
- Inconsistent example structures
- Some ambiguous or noisy examples
- Lack of strong structural validation
- Limited control over idiom semantic metadata

To address these issues and improve dataset quality, **Version 2 introduces a more robust and controlled enrichment pipeline**.

---

# Key Improvements in Version 2

## 1. Structured Output Schema
V2 enforces a **strict JSON schema** for the generated output, ensuring that every generated record contains:

- canonical idiom
- idiom meaning (English)
- idiom meaning (Arabic)
- idiom compositionality
- ambiguity flag
- register and domain
- learner difficulty
- multiple labeled examples

Each idiom produces **8 examples**:
- idiomatic usage
- literal usage (where possible)

This structure supports multiple supervised learning tasks.

---

## 2. Balanced Idiomatic vs Literal Examples

The generation process explicitly enforces a balanced distribution between:

- **idiomatic usage**
- **literal usage**

This balance is critical for training **idiom detection models** that distinguish between figurative and literal contexts.

---

## 3. Batch LLM Processing

The pipeline uses **batch LLM inference**, allowing large-scale dataset generation efficiently while reducing API overhead and ensuring reproducibility.

---

## 4. Automatic Dataset Validation

After enrichment, the dataset passes through a validation stage that checks:

- JSON structure correctness
- example completeness
- presence of idioms in example text
- consistency of labels
- missing fields

Rows failing validation are flagged as:
needs_review

This ensures problematic records are isolated without corrupting the final dataset.

---

## 5. Targeted Verification of Suspicious Rows

Instead of regenerating the entire dataset, V2 introduces a **verification stage** where only rows marked `needs_review` are re-evaluated using the LLM.

This targeted verification process:

- reduces cost
- improves dataset reliability
- corrects ambiguous examples
- preserves valid generated data

---

# Expected Outcomes

The Version 2 pipeline produces a **large, structured, multilingual idiom dataset** with:

- idiom metadata
- canonical meanings
- Arabic translations
- labeled example usage
- balanced literal/idiomatic contexts

The resulting dataset is designed to support **multiple experimental tasks** in the IdiomX research project and serve as a high-quality benchmark for idiom understanding models.

---

# Dataset Characteristics (Expected)

Approximate dataset scale after enrichment:

| Metric | Value |
|------|------|
Total examples | ~120K+
Idiomatic examples | ~50%
Literal examples | ~50%
Languages | English + Arabic
Validation coverage | >98% valid records

This scale significantly exceeds most existing idiom datasets used in prior research.

---

# Pipeline Steps

The Version 2 enrichment pipeline follows these stages:

1. Raw idiom dataset preparation  
2. Batch LLM enrichment  
3. Result downloading  
4. JSON repair and merge  
5. Dataset validation  
6. Suspicious row verification  
7. Final dataset export

The final output becomes the **training dataset used in the IdiomX deep learning experiments**.

---

# Next Stage

After completing this enrichment pipeline, the dataset will be used for training and evaluating several models including:

- Logistic Regression baseline
- XGBoost classifier
- BERT idiom detection model
- RoBERTa idiom detection model
- Retrieval-based idiom matching models

These experiments form the core of the **IdiomX research evaluation pipeline**.

In [3]:
import sys
!{sys.executable} -m pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
from llm_enrichment.config.api_config import client
print("api client import: OK")

api client import: OK


In [4]:
from pathlib import Path
import sys
import importlib

CURRENT_DIR = Path.cwd().resolve()
LLM_ENRICHMENT_ROOT = CURRENT_DIR.parent
IDIOMX_ROOT = LLM_ENRICHMENT_ROOT.parent

# Put IdiomX root first
if str(IDIOMX_ROOT) in sys.path:
    sys.path.remove(str(IDIOMX_ROOT))
sys.path.insert(0, str(IDIOMX_ROOT))

importlib.invalidate_caches()

print("Current dir:", CURRENT_DIR)
print("LLM root:", LLM_ENRICHMENT_ROOT)
print("IdiomX root:", IDIOMX_ROOT)
print("sys.path[0]:", sys.path[0])
print("llm_enrichment exists:", (IDIOMX_ROOT / "llm_enrichment").exists())
print("scripts exists:", (IDIOMX_ROOT / "llm_enrichment" / "scripts").exists())

Current dir: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\notebooks
LLM root: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment
IdiomX root: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
sys.path[0]: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
llm_enrichment exists: True
scripts exists: True


## 1. Define Paths

In [3]:
from pathlib import Path
import sys
import importlib

CURRENT_DIR = Path.cwd().resolve()
IDIOMX_ROOT = CURRENT_DIR.parent
LLM_ENRICHMENT_ROOT = IDIOMX_ROOT / "llm_enrichment"
#IDIOMX_ROOT = LLM_ENRICHMENT_ROOT.parent

# Put IdiomX root first
if str(IDIOMX_ROOT) in sys.path:
    sys.path.remove(str(IDIOMX_ROOT))
sys.path.insert(0, str(IDIOMX_ROOT))

importlib.invalidate_caches()

print("Current dir:", CURRENT_DIR)
print("LLM root:", LLM_ENRICHMENT_ROOT)
print("IdiomX root:", IDIOMX_ROOT)
print("sys.path[0]:", sys.path[0])
print("llm_enrichment exists:", (IDIOMX_ROOT / "llm_enrichment").exists())
print("scripts exists:", (IDIOMX_ROOT / "llm_enrichment" / "scripts").exists())

Current dir: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\notebooks
LLM root: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment
IdiomX root: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
sys.path[0]: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
llm_enrichment exists: True
scripts exists: True


## 2. Define Script files

In [5]:
from llm_enrichment.scripts.prepare_batch_requests_v2 import prepare_batch_requests
from llm_enrichment.scripts.submit_batch_v2 import submit_batch
from llm_enrichment.scripts.check_batch_v2 import check_batch
from llm_enrichment.scripts.download_results_v2 import download_results
from llm_enrichment.scripts.merge_results_v2 import merge_results
from llm_enrichment.scripts.validate_dataset_v2 import validate_dataset
from llm_enrichment.scripts.verify_suspicious_rows_v2 import verify_suspicious_rows

print("All imports OK")

All imports OK


## 3. Prepare batch requests

In [3]:
prepare_batch_requests()

KeyboardInterrupt: 

In [4]:
from pathlib import Path
LLM_ENRICHMENT_ROOT = IDIOMX_ROOT / "llm_enrichment"
batch_file = Path(LLM_ENRICHMENT_ROOT / "data/batches/idiomx_batch_v2.jsonl").resolve()
#batch_file = Path("../data/batches/idiomx_batch_v2.jsonl").resolve()
print(batch_file.exists(), batch_file)

True C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\batches\idiomx_batch_v2.jsonl


## 4. Submit batch

In [11]:
batch_id = submit_batch()
print("Batch ID:", batch_id)

Uploaded file ID: file-KiPnT7d8zwPr5Uh7egmVbo
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Batch status: validating
Batch info saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\batches\idiomx_batch_v2_info.json
Batch ID: None


## 5. check if submitting batc is completed

In [4]:
import time
from llm_enrichment.scripts.check_batch_v2 import check_batch

check_count = 0

while True:
    batch = check_batch()
    status = batch.status
    
    check_count += 1
    print(f"Check {check_count}: {status}")
    
    if status in ["completed", "failed", "cancelled", "expired"]:
        print("Batch finished with status:", status)
        break
    
    time.sleep(60)

Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 1: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 2: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 3: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 4: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 5: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: in_progress
Created at: 1773645372
Output file ID: None
Error file ID: None
Check 6: in_progress
Batch ID: batch_69b7ae3c528081909b0b8fb6d689270f
Status: finalizing
Created 

## 6. Download results

In [5]:
from llm_enrichment.scripts.download_results_v2 import download_results
download_results()

Downloaded results to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\results\idiomx_results_v2.jsonl


WindowsPath('C:/Users/ayman/Documents/IdiomX/github_idiomX/IdiomX/llm_enrichment/data/results/idiomx_results_v2.jsonl')

### json repair installation (final json has brokken lines)

In [7]:
import sys
!{sys.executable} -m pip install json-repair

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import importlib
import llm_enrichment.scripts.merge_results_v2 as merge_mod

importlib.reload(merge_mod)
from llm_enrichment.scripts.merge_results_v2 import merge_results

In [3]:
# check the json file for broken lines
import json
from pathlib import Path

results_path = Path(LLM_ENRICHMENT_ROOT / "data/results/idiomx_results_v2.jsonl").resolve()

bad_id = "idiomx_v2_65"

with open(results_path, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec.get("custom_id") == bad_id:
            body = rec["response"]["body"]

            output_text = None
            for item in body.get("output", []):
                if item.get("type") == "message":
                    for c in item.get("content", []):
                        if c.get("type") == "output_text":
                            output_text = c.get("text")
                            break

            print(output_text[:4000])
            break

## 7. merge results

In [7]:
merge_results()

Skipping idiomx_v2_65: invalid JSON - Expecting ',' delimiter: line 1 column 2083 (char 2082)
Skipping idiomx_v2_737: invalid JSON - Expecting ',' delimiter: line 1 column 2413 (char 2412)
Skipping idiomx_v2_1894: invalid JSON - Expecting ',' delimiter: line 1 column 3663 (char 3662)
Skipping idiomx_v2_3331: invalid JSON - Expecting ',' delimiter: line 1 column 4101 (char 4100)
Skipping idiomx_v2_4089: invalid JSON - Expecting ',' delimiter: line 1 column 4055 (char 4054)
Skipping idiomx_v2_5099: invalid JSON - Expecting ',' delimiter: line 1 column 4139 (char 4138)
Skipping idiomx_v2_7744: invalid JSON - Expecting ',' delimiter: line 1 column 3793 (char 3792)
Skipping idiomx_v2_12168: invalid JSON - Expecting ',' delimiter: line 1 column 1490 (char 1489)
Skipping idiomx_v2_15368: invalid JSON - Unterminated string starting at: line 1 column 2442 (char 2441)
Saved merged v2 dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\enriched\idiomx_enriched_v2.

,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url,idiom_canonical,...,idiom_in_example,idiom_in_example_arabic,idiom_in_example_meaning_en,idiom_in_example_meaning_arabic,is_example_idiom,example_usage_label,is_generated_example,enrichment_model,enrichment_version,validation_status
0,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,kaikki_wiktionary,dictionary,noun,NaN,high,NaN,$100 hamburger,...,"Last weekend, Mark took a $100 hamburger trip ...","في نهاية الأسبوع الماضي، قام مارك برحلة ""همبرغ...",The expression figuratively means that Mark fl...,يعني التعبير مجازياً أن مارك طار فقط للمتعة ول...,True,idiomatic,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
1,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,kaikki_wiktionary,dictionary,noun,NaN,high,NaN,$100 hamburger,...,Pilots often joke about taking a $100 hamburge...,"غالبًا ما يمزح الطيارون بشأن القيام برحلة ""همب...",Used figuratively to describe a casual flight ...,تُستخدم مجازياً لوصف رحلة عادية تُقام أساسًا ل...,True,idiomatic,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
2,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,kaikki_wiktionary,dictionary,noun,NaN,high,NaN,$100 hamburger,...,Her $100 hamburger flight was an excuse for a ...,كانت رحلة الهمبرغر بمئة دولار مبررًا لعطلة قصي...,"Figuratively, the flight served as a recreatio...",مجازياً، كانت الرحلة بمثابة رحلة ترفيهية مع اس...,True,idiomatic,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
3,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,kaikki_wiktionary,dictionary,noun,NaN,high,NaN,$100 hamburger,...,Taking a $100 hamburger is a lighthearted trad...,"يُعتبر القيام برحلة ""همبرغر بمئة دولار"" تقليدً...","A playful way to say pilots fly for fun, using...",طريقة مرحة لوصف طيران الطيارين للمتعة باستخدام...,True,idiomatic,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
4,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,kaikki_wiktionary,dictionary,noun,NaN,high,NaN,$100 hamburger,...,The restaurant claims their hamburger costs $1...,تدعي المطعم أن الهمبرغر الخاص بهم يكلف 100 دول...,Literal use indicating that the price of the h...,استخدام حرفي يشير إلى أن سعر الهمبرغر هو فعلاً...,False,literal,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128779,éminence grise,A secret or unofficial decision-maker.,But Harry Hopkins was no mere manipulator of p...,kaikki_wiktionary,dictionary,noun,idiomatic,high,NaN,eminence grise,...,Many believe the CEO’s father is the eminence ...,يعتقد كثيرون أن والد الرئيس التنفيذي هو العلام...,A figurative expression for an unofficial but ...,تعبير مجازي عن مستشار غير رسمي لكنه مؤثر في سي...,True,idiomatic,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
128780,éminence grise,A secret or unofficial decision-maker.,But Harry Hopkins was no mere manipulator of p...,kaikki_wiktionary,dictionary,noun,idiomatic,high,NaN,eminence grise,...,The castle’s tower offered a clear view from t...,كان برج القلعة يوفر رؤية واضحة من السلسلة الجب...,Describes a literal gray-colored hill or promi...,يصف تلة رمادية اللون بارزة تطل على الوادي.,False,literal,1,gpt-4.1-mini-2025-04-14,v2_4idiomatic_4literal,pending
128781,éminence grise,A secret or unofficial decision-maker.,But Harry Hopkins was no mere manipulator of p...,kaikki_wiktionary,dictionary,noun,idiomatic,high,NaN,eminence grise,...,"Artists gathered at the park’s eminence grise,...",تجمع الفنانون في العلامة الجبلية الرمادية في ا...,The phrase used literally to denote a physical...,استخدام العبارة حرفياً للدلالة على معلم مادي م...,False,literal,1,gpt-

## 8. Validate dataset

In [8]:
from llm_enrichment.scripts.validate_dataset_v2 import validate_dataset

validate_dataset()

C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\scripts\validate_dataset_v2.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


Validated dataset saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\enriched\idiomx_enriched_v2_validated.csv
Issues saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\enriched\idiomx_enriched_v2_issues.csv
validation_status
valid           126429
needs_review      2355
Name: count, dtype: int64


## 9. Verify suspicious records (and fix them)

## review and fix the needs_review rows

In [6]:
from llm_enrichment.scripts.verify_suspicious_rows_v2 import verify_suspicious_rows

verify_suspicious_rows()

100%|████████████████████████████████████████████████████████████████████████████| 2355/2355 [4:56:03<00:00,  7.54s/it]


Saved final v2 dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\llm_enrichment\data\enriched\idiomx_enriched_v2_final.csv
validation_status
valid        126429
verified       1617
corrected       738
Name: count, dtype: int64
